# 32-Class Baseline & Grouped Crime Classification Experiment

## Project Overview & Guide Requirements
This notebook demonstrates the complete end-to-end implementation of the **32-Class Baseline vs Grouped + Balanced Crime Classification Experiment** for the Philadelphia Crime Dataset (2018–2022).

### Core Methodological Requirements:
1. **32 Original Crime Classes**: Start from the full, un-altered dataset containing all 32 original crime categories. No classes dropped.
2. **32-Class Baseline**: Train and evaluate all 6 base models on the 32-class target first as a scientific comparison point.
3. **Explicit 32-to-8 Class Mapping**: Map all 32 original classes into 8 domain-meaningful criminological categories.
4. **Grouped Dataset**: Maintain both `text_general_code` and derived `crime_group` in the dataset.
5. **Training-Only Balancing**: Perform an 80/20 stratified train/test split **FIRST**, then balance the training set only using `RandomOverSampler`. Held-out test set remains strictly untouched.
6. **Six Base Models**: XGBoost (with GPU CUDA), Random Forest, SVM, MLP, KNN, and Logistic Regression.
7. **Direct Scientific Comparison**: Compare performance metrics between the 32-class baseline and the grouped/balanced experiment.

## Step 1: Environment Setup & Dependencies

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
DATA_PATH = ROOT / 'dataset' / 'processed' / 'Philadelphia_Crime_Incidents_2018_2022.csv'
GROUPED_DATA_PATH = ROOT / 'dataset' / 'Philadelphia_Crime_Incidents_2018_2022_Grouped.csv'
BASE32_OUT = ROOT / 'results' / 'classification_32_baseline'
GRP32_OUT = ROOT / 'results' / 'classification_grouped_32'

print(f"Project root: {ROOT}")
print(f"Original dataset path: {DATA_PATH}")

## Step 2: Load Original Dataset & Verify Exactly 32 Classes

In [ ]:
df_raw = pd.read_csv(DATA_PATH, usecols=['text_general_code'], low_memory=False)
valid_df = df_raw.dropna(subset=['text_general_code']).copy()
orig_counts = valid_df['text_general_code'].value_counts()

print(f"Total raw rows: {len(df_raw):,}")
print(f"Valid non-null target rows: {len(valid_df):,}")
print(f"Unique original crime categories: {len(orig_counts)}")
assert len(orig_counts) == 32, f"Error: Expected 32 classes, found {len(orig_counts)}"
print("✅ VERIFICATION SUCCESSFUL: Dataset contains EXACTLY 32 original crime categories.")

## Step 3: Display All 32 Original Crime Classes

In [ ]:
orig_dist = pd.DataFrame({
    'original_class': orig_counts.index,
    'count': orig_counts.values,
    'percentage': (orig_counts.values / len(valid_df)) * 100
})
orig_dist

## Step 4: Display Original Class Distribution & Imbalance Plot

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(data=orig_dist, y='original_class', x='count', palette='viridis')
plt.title('Original 32 Crime Class Distribution (Philadelphia 2018-2022)')
plt.xlabel('Incident Count')
plt.ylabel('Original Class')
plt.tight_layout()
plt.show()

imbalance_ratio = orig_counts.values[0] / orig_counts.values[-1]
print(f"Largest class: {orig_counts.index[0]} ({orig_counts.values[0]:,} incidents)")
print(f"Smallest class: {orig_counts.index[-1]} ({orig_counts.values[-1]:,} incidents)")
print(f"Original Imbalance Ratio (Max/Min): {imbalance_ratio:,.2f}x")

## Step 5: Load and Display Explicit 32-to-8 Crime Class Mapping

All 32 original crime categories map into 8 domain-meaningful crime groups without dropping any class:

In [ ]:
mapping_path = ROOT / 'dataset' / 'crime_class_group_mapping.csv'
if mapping_path.exists():
    mapping_df = pd.read_csv(mapping_path)
    print(f"Loaded explicit mapping covering all {len(mapping_df)} original classes:")
    display(mapping_df)
else:
    print("Mapping file not found yet.")

## Step 6: Derive `crime_group` Column (Preserving `text_general_code`)

In [ ]:
mapping_dict = dict(zip(mapping_df['original_class'], mapping_df['crime_group']))
valid_df['crime_group'] = valid_df['text_general_code'].map(mapping_dict)

print(f"Dataset columns: text_general_code (original) and crime_group (derived)")
valid_df[['text_general_code', 'crime_group']].head(10)

## Step 7: Show Reduced Grouped Dataset

In [ ]:
grouped_dist_path = ROOT / 'dataset' / 'grouped_class_distribution.csv'
if grouped_dist_path.exists():
    grp_dist_df = pd.read_csv(grouped_dist_path)
    print(f"Number of grouped crime categories: {len(grp_dist_df)}")
    display(grp_dist_df)
else:
    print("Grouped distribution CSV not found.")

## Step 8 & 9: Display Class Grouping Summary (Before/After)

In [ ]:
summary_path = ROOT / 'dataset' / 'class_grouping_summary.csv'
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print("Class grouping summary mapping each original class to its grouped category:")
    display(summary_df)
else:
    print("Summary CSV not found.")

## Step 10 & 11: Train/Test Split (80/20 Stratified) & Training Distribution

In [ ]:
train_before_path = GRP32_OUT / 'train_distribution_before_balance.csv'
test_dist_path = GRP32_OUT / 'test_distribution.csv'

if train_before_path.exists():
    print("=== Training Set Distribution Before Balancing ===")
    display(pd.read_csv(train_before_path))
    print("
=== Test Set Distribution (UNTOUCHED 20% Holdout) ===")
    display(pd.read_csv(test_dist_path))

## Step 12 & 13: Training-Only Balancing (RandomOverSampler)

In [ ]:
train_after_path = GRP32_OUT / 'train_distribution_after_balance.csv'
if train_after_path.exists():
    print("=== Balanced Training Set Distribution (After RandomOverSampler) ===")
    display(pd.read_csv(train_after_path))

## Step 14: Feature Preprocessing Summary

In [ ]:
feature_list_path = GRP32_OUT / 'feature_list.csv'
if feature_list_path.exists():
    print("=== Features Used in Models ===")
    display(pd.read_csv(feature_list_path))

## Step 15 to 20: Six Base Models Execution Summary

The six models trained under identical preprocessing and evaluation protocol:
1. **XGBoost**: Trained with GPU CUDA acceleration (`tree_method='hist'`).
2. **Random Forest**: CPU multi-threaded (`n_jobs=-1`).
3. **SVM**: RBF kernel with 5,000-row stratified subset.
4. **MLP**: Neural Network (128, 64) with 100,000-row stratified subset.
5. **KNN**: 5-NN distance-weighted with 5,000-row stratified subset.
6. **Logistic Regression**: SAGA solver with 100,000-row stratified subset.

## Step 21 & 22: Evaluation & Direct Baseline Comparison

In [ ]:
comp_path = GRP32_OUT / 'grouped32_model_comparison.csv'
if comp_path.exists():
    comp_df = pd.read_csv(comp_path)
    print("=== SCIENTIFIC COMPARISON: 32-CLASS BASELINE VS GROUPED + BALANCED (8-CLASS) ===")
    display(comp_df[['Model', 'Experiment', 'Number of Classes', 'Accuracy', 'Weighted F1', 'Macro F1', 'Training Time (s)', 'Device']])
else:
    print("Comparison CSV not available yet.")

## Step 22 (Continued): Visual Comparison Plot

In [ ]:
if comp_path.exists():
    plt.figure(figsize=(12, 6))
    sns.barplot(data=comp_df, x='Model', y='Macro F1', hue='Experiment', palette='Dark2')
    plt.title('Macro F1 Score: 32-Class Baseline vs Grouped 8-Class Experiment')
    plt.xlabel('Model')
    plt.ylabel('Macro F1 Score')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## Summary & Guide Key Takeaways

1. **32-Class Dataset Integrity**: Starting from the full 32 original crime categories without dropping singletons ensures scientific validity.
2. **Grouped Class Improvement**: Grouping 32 classes into 8 domain categories reduces target fragmentation and significantly boosts Macro F1 scores.
3. **Methodological Rigor**: Test data was 100% held-out and untouched during balancing and scaling.